# Nesterov Accelerated Gradient — A method for convex minimization with rate O(1/k²) (1983)

_Papers · Foundations & Optimization_

**Look one momentum step ahead before measuring the slope — and a smooth convex problem's error falls like 1/k² instead of gradient descent's 1/k.**

---

This notebook is a practice scaffold for the **Nesterov Accelerated Gradient — A method for convex minimization with rate O(1/k²) (1983)** lesson. We build it one piece at a time: first match a from-scratch optimizer against PyTorch's, then race the accelerated method on a smooth convex quadratic to see the 1/k² rate appear. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Build the look-ahead optimizer from scratch

Nesterov's trick is to measure the slope **one momentum step ahead** of where you stand, not at your current point. PyTorch stores this in an algebraically-equivalent buffer form: it keeps a velocity buffer `buf`, blends in the fresh gradient, then steps along `g + mu*buf`. We implement that exact form so our optimizer will match `torch.optim.SGD(nesterov=True)` bit-for-bit.

In [ ]:
import torch

torch.manual_seed(0)

class MyNesterovSGD:
    """Nesterov Accelerated Gradient from scratch.
       Look-ahead form (Nesterov 1983; Sutskever et al. 2013, eqs. 3-4):
           v <- mu*v - lr*grad(theta + mu*v);  theta <- theta + v
       Implemented in PyTorch's algebraically-equivalent buffer form so torch.allclose holds:
           buf <- mu*buf + g ;  d <- g + mu*buf ;  theta <- theta - lr*d ."""
    def __init__(self, params, lr, momentum):
        self.params = list(params)
        self.lr, self.mu = lr, momentum
        self.buf = [None] * len(self.params)          # velocity buffer per parameter

    @torch.no_grad()
    def step(self):
        for i, p in enumerate(self.params):
            g = p.grad
            if self.buf[i] is None:
                self.buf[i] = g.clone()               # PyTorch seeds the buffer with the first gradient
            else:
                self.buf[i].mul_(self.mu).add_(g)     # buf = mu*buf + g
            d = g + self.mu * self.buf[i]             # Nesterov look-ahead gradient direction
            p.add_(d, alpha=-self.lr)                 # theta -= lr * d

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

### Step 2 — Check it against PyTorch's official Nesterov SGD

A from-scratch optimizer is only trustworthy if it matches a reference. We start two identical weight tensors, drive one with our `MyNesterovSGD` and the other with `torch.optim.SGD(nesterov=True)`, and run the *same* least-squares problem through both for 8 steps. If our buffer math is right, the two weight tensors stay equal to floating-point precision.

In [ ]:
# ---- THE ORACLE: MyNesterovSGD must equal torch.optim.SGD(nesterov=True) over several steps ----
w_mine = torch.randn(5, 3, requires_grad=True)
w_ref  = w_mine.detach().clone().requires_grad_(True)         # identical start
opt_mine = MyNesterovSGD([w_mine], lr=0.05, momentum=0.9)
opt_ref  = torch.optim.SGD([w_ref], lr=0.05, momentum=0.9, nesterov=True)

X = torch.randn(20, 5)
target = torch.randn(20, 3)

for _ in range(8):
    # one step of our optimizer
    opt_mine.zero_grad()
    loss_mine = (((X @ w_mine) - target) ** 2).mean()
    loss_mine.backward()
    opt_mine.step()

    # one step of the reference optimizer on the identical problem
    opt_ref.zero_grad()
    loss_ref = (((X @ w_ref) - target) ** 2).mean()
    loss_ref.backward()
    opt_ref.step()

matches = torch.allclose(w_mine, w_ref, atol=1e-6)
max_abs_diff = (w_mine - w_ref).abs().max().item()
print("allclose vs torch SGD(nesterov=True) after 8 steps:", matches)   # expect True
print("max abs diff:", max_abs_diff)                                    # ~3e-8

### Step 3 — Trace the look-ahead by hand on f(x) = x²

To *see* the look-ahead, run the scheme by hand on the simplest convex bowl, `f(x) = x²`, whose gradient is `2x`. Each step we first hop to the look-ahead point `theta + mu*v`, measure the slope **there**, update the velocity, then move. The printed trajectory should match the worked example: `theta` lands on 1.6000 after step 1 and 0.9920 after step 2.

In [ ]:
# ---- recompute the two-step worked example on f(x)=x^2, lr=0.1, mu=0.9, theta0=2, v0=0 ----
theta = 2.0
v = 0.0
lr = 0.1
mu = 0.9

for t in (1, 2):
    look = theta + mu * v            # look-ahead point
    g_look = 2 * look                # grad of x^2 at the look-ahead point
    v = mu * v - lr * g_look         # velocity update
    theta = theta + v                # parameter update
    print(f"step {t}: look={look:.4f} grad={g_look:.4f} v={v:.4f} theta={theta:.4f}")
# expect: step 1 -> theta=1.6000 ; step 2 -> theta=0.9920

## Visualize the data & results

_On the same smooth convex quadratic from the same start, does Nesterov's accelerated gradient pull AWAY from plain gradient descent as steps grow (the signature of O(1/k^2) beating O(1/k)), and does the look-ahead beat heavy-ball momentum at matched mu and learning rate?_

### Step 1 — Set up a smooth convex quadratic

We need a problem where the *rate* of convergence is visible. A quadratic `f(x) = ½ xᵀAx` with eigenvalues spread from 0 to 1 is smooth and convex but **not** strongly convex (its smallest eigenvalue is 0), which is exactly the regime where the polynomial rates O(1/k) and O(1/k²) show up. We fix the step from the largest eigenvalue `L = 1`.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# smooth convex (NOT strongly convex) quadratic: min eigenvalue 0 exposes the polynomial rates
D = 100
eig = np.linspace(0.0, 1.0, D)
eig[0] = 0.0                          # force the smallest eigenvalue to exactly 0
A = np.diag(eig)
L = 1.0                               # Lipschitz constant of the gradient (largest eigenvalue)

def f(x):
    return 0.5 * x @ (A @ x)

def grad(x):
    return A @ x

x0 = rng.normal(0, 1, D)              # shared random start for every method

### Step 2 — Define gradient descent and Nesterov's accelerated scheme

Plain gradient descent just steps downhill at the current point. Nesterov's constant-step scheme keeps a second sequence `y` and an evolving momentum coefficient `gamma` derived from the `lam` recursion — that extrapolation is what upgrades the rate from O(1/k) to O(1/k²). Both record `f(x)` at every step so we can compare their error curves.

In [ ]:
def gd(steps):
    x = x0.copy()
    out = []
    for k in range(steps):
        out.append(f(x))
        x = x - (1.0 / L) * grad(x)
    return out

def nag(steps):                       # Nesterov constant-step scheme, smooth convex
    x = x0.copy()
    y = x0.copy()
    lam = 0.0
    out = []
    for k in range(steps):
        out.append(f(x))
        xn = y - (1.0 / L) * grad(y)              # gradient step from the extrapolated point y
        lam_n = (1 + np.sqrt(1 + 4 * lam**2)) / 2  # the classic lambda recursion
        gamma = (1 - lam) / lam_n                  # extrapolation weight
        y = (1 - gamma) * xn + gamma * x           # look ahead for the next step
        x, lam = xn, lam_n
    return out

### Step 3 — Add the matched-knob momentum ablation

To isolate *why* Nesterov wins, we also write a single momentum routine with a `nesterov` flag. Heavy-ball and Nesterov share the same `mu` and learning rate; the **only** difference is where the gradient is measured — at `x` (heavy-ball) or at the look-ahead `x + mu*v` (Nesterov). Holding everything else fixed attributes any gap to the look-ahead alone.

In [ ]:
def momentum(steps, nesterov):        # ablation: same mu, lr; only the look-ahead changes
    x = x0.copy()
    v = np.zeros(D)
    mu = 0.9
    lr = 0.5 / L
    out = []
    for k in range(steps):
        out.append(f(x))
        if nesterov:
            g = grad(x + mu * v)      # measure the slope at the look-ahead point
        else:
            g = grad(x)               # heavy-ball measures at the current point
        v = mu * v - lr * g
        x = x + v
    return out

### Step 4 — Run the race and read the rates

Now run all four (GD, NAG, and the two momentum variants) for 300 steps. Watch the GD/NAG ratio blow up as `k` grows — that widening gap is the O(1/k) vs O(1/k²) signature pulling apart. The ablation line confirms the look-ahead, at matched `mu` and learning rate, reaches a far lower loss than heavy-ball.

In [ ]:
g = gd(300)
n = nag(300)
print("GD  f-f* at k=10,100,300:", [round(g[i], 10) for i in (9, 99, 299)])
print("NAG f-f* at k=10,100,300:", [round(n[i], 12) for i in (9, 99, 299)])
print("ratio GD/NAG at k=300:", round(g[299] / n[299], 1))          # ~350x

hb = momentum(300, nesterov=False)
na = momentum(300, nesterov=True)
print("ablation at k=50  heavy-ball:", round(hb[49], 6), " nesterov:", round(na[49], 6))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Do one heavy-ball (classical momentum) step on $f(x)=x^2$ from $\theta_0=2,\,v_0=0$ with $\varepsilon=0.1,\ \mu=0.9$, and confirm it equals the NAG step 1 above. Then explain why step 2 will differ.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- CM gradient at the current point: $\nabla f(2.0)=2\cdot2=4.0$. — _Heavy-ball measures at $\theta_t$, not the look-ahead._
- $v_1=0.9\cdot0-0.1\cdot4.0=-0.4$; $\theta_1=2.0-0.4=1.6$. — _Same as NAG because $v_0=0$ makes the look-ahead point equal the current point._
- At step 2, $v_1=-0.4\ne0$, so NAG looks ahead to $1.24$ while CM stays at $1.6$. — _Once velocity is nonzero the two methods see different gradients._

**Answer:** Step 1 is identical ($\theta_1=1.6$) because with $v_0=0$ the look-ahead point $\theta_0+\mu v_0$ equals $\theta_0$. From step 2 on they diverge: NAG measures the slope at $\theta_1+\mu v_1=1.24$ (gradient $2.48$), CM at $\theta_1=1.6$ (gradient $3.2$). NAG's look-ahead already feels that it is approaching the minimum and eases off, which is the source of its better conditioning and faster convergence.

</details>

**Problem 2.** Acceleration arithmetic: if gradient descent needs about $1/\delta$ steps to reach error $\delta$ on a smooth convex problem, roughly how many does NAG need, and by what factor is that fewer at $\delta=10^{-4}$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- GD: $f-f^\star\le c/k\Rightarrow k\approx c/\delta$. — _$O(1/k)$ rate inverted._
- NAG: $f-f^\star\le c'/k^2\Rightarrow k\approx \sqrt{c'/\delta}=O(1/\sqrt{\delta})$. — _$O(1/k^2)$ rate inverted._
- Ratio $\approx (1/\delta)/(1/\sqrt{\delta})=1/\sqrt{\delta}$. At $\delta=10^{-4}$, $1/\sqrt{\delta}=100$. — _Square-root fewer steps._

**Answer:** NAG needs about $1/\sqrt{\delta}$ steps versus GD's $1/\delta$ — a factor of $1/\sqrt{\delta}$ fewer. At $\delta=10^{-4}$ that is roughly $100\times$ fewer iterations (ignoring constants). This square-root speedup is exactly the practical meaning of going from $O(1/k)$ to $O(1/k^2)$.

</details>

**Problem 3.** Ablation: in the CODEVIZ, run heavy-ball momentum and NAG with the SAME $\mu$ and learning rate on the smooth convex quadratic. What do you expect, and what does our run show?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Heavy-ball measures $\nabla f(\theta_t)$; NAG measures $\nabla f(\theta_t+\mu v_t)$. — _Only the gradient's evaluation point changes._
- NAG's earlier correction damps the oscillation heavy-ball shows along high-curvature directions. — _Sutskever et al. 2013, Section 2.1 / Figure 1._
- Compare the loss at matched steps. — _Same $\mu$, lr isolates the look-ahead's effect._

**Answer:** With everything else held fixed, the only change is where the gradient is measured, so any difference is the look-ahead's doing. In our small run NAG reaches a markedly lower loss than heavy-ball at the same step (e.g. ~1.7e-4 vs ~8.3e-2 by step 50) and oscillates less — the look-ahead corrects the velocity before it overshoots. This is the qualitative effect the paper describes, reproduced on toy data (our numbers, not the paper's).

</details>